# Stage 7: Advanced Forecasting

Compare saved seasonal-naive and linear-regression baselines with compact Random Forest, HistGradientBoosting, and seasonal SARIMAX models. All series retain separate Entry/Exit targets and chronological Stage 4 partitions. Tree model selection is validation-only; the held-out test is evaluated once. Models with weather use observed prediction-hour weather, so those results are retrospective unless forecast weather is supplied.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from statsmodels.tsa.statespace.sarimax import SARIMAX

project_root = Path.cwd().resolve()
if project_root.name == 'Notebooks':
    project_root = project_root.parent
data_dir = project_root / 'DataProcessed'
target = 'ridership_count'
splits = {name: pd.read_csv(data_dir / f'metro_weather_{name}_prepared.csv', parse_dates=['time']) for name in ['train', 'validation', 'test']}
for frame in splits.values():
    assert frame[['time', 'direction', target]].notna().all().all()
    assert not frame.duplicated(['time', 'direction']).any()
    assert set(frame.direction) == {'Entry', 'Exit'}
train = splits['train'].sort_values(['time', 'direction']).reset_index(drop=True)
validation = splits['validation'].sort_values(['time', 'direction']).reset_index(drop=True)
test = splits['test'].sort_values(['time', 'direction']).reset_index(drop=True)
assert train.time.max() < validation.time.min() < test.time.min()
assert train.groupby('time').direction.nunique().eq(2).all() and validation.groupby('time').direction.nunique().eq(2).all() and test.groupby('time').direction.nunique().eq(2).all()
full_target = pd.read_csv(data_dir / 'metro_weather_features_final.csv', parse_dates=['time'])
assert not full_target.duplicated(['time', 'direction']).any()
print('Split rows:', {name: len(frame) for name, frame in splits.items()}, '| chronological and direction checks passed.')

Split rows: {'train': 3780, 'validation': 810, 'test': 810} | chronological and direction checks passed.


## Model Setup

Model A uses direction, temporal cycles/indicators, and historical ridership. Model B adds weather/rainfall. Redundant precipitation/rain encodings, apparent temperature, the correlated 3h rolling mean, target, and raw timestamps are excluded. Imputation and direction encoding are fitted inside each training-only pipeline. SARIMAX uses a regular hourly grid with non-operating hours as missing endog values, never zero; it is a univariate no-weather comparator.

In [2]:
# Load saved Stage 5/6 baseline forecasts; never refit or select using test scores.
baseline = pd.read_csv(data_dir / 'baseline_forecasting_predictions.csv', parse_dates=['time'])
ablation = pd.read_csv(data_dir / 'feature_ablation_predictions.csv', parse_dates=['time'])
assert set(baseline.split) == {'validation', 'test'} and set(ablation.split) == {'validation', 'test'}
baseline_specs = {
    'SeasonalNaive_24h': (baseline, 'seasonal_naive_24h'),
    'LinearRegression_B_temporal_weather': (ablation, 'B_plus_weather'),
    'LinearRegression_A_temporal_history': (ablation, 'A_temporal_history')
}
baseline_predictions = []
for output_name, (source, source_model) in baseline_specs.items():
    for split_name in ['validation', 'test']:
        selected = source.loc[source.split.eq(split_name) & source.model.eq(source_model), ['time', 'direction', 'actual', 'prediction']].copy()
        selected['split'] = split_name
        selected['model'] = output_name
        baseline_predictions.append(selected)
baseline_predictions = pd.concat(baseline_predictions, ignore_index=True)
for split_name, frame in [('validation', validation), ('test', test)]:
    part = baseline_predictions[baseline_predictions.split.eq(split_name)]
    expected = frame[['time', 'direction', target]].rename(columns={target: 'actual'})
    for _, group in part.groupby('model'):
        aligned = group.merge(expected, on=['time', 'direction', 'actual'], how='left', validate='one_to_one')
        assert len(aligned) == len(frame) and aligned.actual.notna().all() and group.prediction.notna().all()
print('Reused saved baseline predictions:', baseline_predictions.model.unique().tolist())

Reused saved baseline predictions: ['SeasonalNaive_24h', 'LinearRegression_B_temporal_weather', 'LinearRegression_A_temporal_history']


In [3]:
temporal = ['direction', 'month', 'is_peak_hour', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
history = ['ridership_lag_1h', 'ridership_lag_2h', 'ridership_lag_3h', 'ridership_lag_24h', 'ridership_rolling_mean_6h', 'ridership_rolling_mean_24h']
weather = ['temperature_2m_c', 'relative_humidity_2m_pct', 'rain_mm', 'weather_code_wmo', 'surface_pressure_hpa', 'cloud_cover_pct', 'visibility_m', 'wind_speed_10m_kmh', 'rainfall_lag_1h', 'rainfall_lag_2h', 'rainfall_lag_3h', 'rainfall_sum_3h', 'rainfall_sum_6h', 'rainfall_sum_12h', 'rainfall_sum_24h']
features_A = temporal + history
features_B = features_A + weather
assert target not in features_B and 'time' not in features_B
numeric_A = [column for column in features_A if column != 'direction']
numeric_B = [column for column in features_B if column != 'direction']
def make_pipeline(numeric_columns, estimator):
    prep = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric_columns),
        ('direction', OneHotEncoder(handle_unknown='ignore'), ['direction'])
    ])
    return Pipeline([('preprocess', prep), ('model', estimator)])

def metric_values(actual, prediction):
    actual = np.asarray(actual, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    error = prediction - actual
    nonzero = actual != 0
    mape = np.mean(np.abs(error[nonzero] / actual[nonzero])) * 100 if nonzero.any() else np.nan
    return {'n': len(actual), 'n_nonzero_mape': int(nonzero.sum()), 'MAE': float(np.mean(np.abs(error))), 'RMSE': float(np.sqrt(np.mean(error ** 2))), 'MAPE_pct': float(mape)}

candidate_specs = {}
for family in ['RandomForest', 'HistGradientBoosting']:
    for variant, features in [('A_temporal_history', features_A), ('B_plus_weather', features_B)]:
        for option in [0, 1]:
            if family == 'RandomForest':
                estimator = RandomForestRegressor(n_estimators=180, max_depth=[14, None][option], min_samples_leaf=[2, 4][option], max_features=0.8, n_jobs=-1, random_state=42)
            else:
                estimator = HistGradientBoostingRegressor(max_iter=[140, 220][option], max_leaf_nodes=[15, 15][option], learning_rate=0.08, l2_regularization=1.0, random_state=42)
            model_name = f'{family}_{variant}'
            pipeline = make_pipeline(numeric_A if variant.startswith('A_') else numeric_B, estimator)
            pipeline.fit(train[features], train[target])
            prediction = pipeline.predict(validation[features])
            candidate_specs[(model_name, option)] = {'features': features, 'pipeline': pipeline, 'prediction': prediction, 'metrics': metric_values(validation[target], prediction), 'option': option}
selected_candidates = {}
tree_validation = []
for model_name in sorted({key[0] for key in candidate_specs}):
    choices = [(key, value) for key, value in candidate_specs.items() if key[0] == model_name]
    best_key, best = min(choices, key=lambda item: (item[1]['metrics']['MAE'], item[1]['metrics']['RMSE']))
    selected_candidates[model_name] = best
    tree_validation.append({'model': model_name, **best['metrics'], 'selected_option': best['option']})
    validation = validation.copy()
    validation[model_name] = best['prediction']
tree_validation_metrics = pd.DataFrame(tree_validation)
assert validation.groupby('time').direction.nunique().eq(2).all()
assert all(np.isfinite(validation[name]).all() for name in selected_candidates)
print('Best validation-tuned tree configurations:')
display(tree_validation_metrics)

Best validation-tuned tree configurations:


,model,n,n_nonzero_mape,MAE,RMSE,MAPE_pct,selected_option
0,HistGradientBoosting_A_temporal_history,810,810,6271.642787,9789.255688,77.783335,1
1,HistGradientBoosting_B_plus_weather,810,810,6751.119987,10088.203082,46.240097,0
2,RandomForest_A_temporal_history,810,810,5849.316393,10107.701423,4.676647,0
3,RandomForest_B_plus_weather,810,810,5778.637351,10106.005026,4.735941,0


In [4]:
# Seasonal AR(1)xAR(1,24) per direction. Missing operating hours are NaN endog; forecasts are one-step ahead, then observed ridership updates state.
def hourly_endog(frame, direction, start, end):
    grid = pd.date_range(start=start, end=end, freq='h')
    observed = frame.loc[frame.direction.eq(direction)].set_index('time')[target]
    return observed.reindex(grid).astype(float)

sarimax_validation = validation[['time', 'direction', target]].copy()
sarimax_test = test[['time', 'direction', target]].copy()
sarimax_validation['SARIMAX_no_weather'] = np.nan
sarimax_test['SARIMAX_no_weather'] = np.nan
train_start, train_end = train.time.min(), train.time.max()
validation_end, test_end = validation.time.max(), test.time.max()
for direction in ['Entry', 'Exit']:
    train_y = hourly_endog(train, direction, train_start, train_end)
    validation_y = hourly_endog(validation, direction, train_end + pd.Timedelta(hours=1), validation_end)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        train_fit = SARIMAX(train_y, order=(1, 0, 0), seasonal_order=(1, 0, 0, 24), trend='c', enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=50)
    validation_fit = train_fit.append(validation_y, refit=False)
    val_mean = validation_fit.get_prediction(start=validation_y.index[0], end=validation_y.index[-1], dynamic=False).predicted_mean
    val_values = val_mean.reindex(validation.loc[validation.direction.eq(direction), 'time']).to_numpy()
    sarimax_validation.loc[sarimax_validation.direction.eq(direction), 'SARIMAX_no_weather'] = val_values
    train_validation_y = hourly_endog(pd.concat([train, validation]), direction, train_start, validation_end)
    test_y = hourly_endog(test, direction, validation_end + pd.Timedelta(hours=1), test_end)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        train_validation_fit = SARIMAX(train_validation_y, order=(1, 0, 0), seasonal_order=(1, 0, 0, 24), trend='c', enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=50)
    test_fit = train_validation_fit.append(test_y, refit=False)
    test_mean = test_fit.get_prediction(start=test_y.index[0], end=test_y.index[-1], dynamic=False).predicted_mean
    test_values = test_mean.reindex(test.loc[test.direction.eq(direction), 'time']).to_numpy()
    sarimax_test.loc[sarimax_test.direction.eq(direction), 'SARIMAX_no_weather'] = test_values
assert np.isfinite(sarimax_validation.SARIMAX_no_weather).all() and np.isfinite(sarimax_test.SARIMAX_no_weather).all()
print('SARIMAX fit in regular hourly time with missing off-hour observations; no weather exogenous variables.')

SARIMAX fit in regular hourly time with missing off-hour observations; no weather exogenous variables.


In [7]:
# After validation selection, refit chosen tree configurations on train+validation, then predict test once.
train_validation = pd.concat([train, validation], ignore_index=True).sort_values(['time', 'direction']).reset_index(drop=True)
for model_name, selected in selected_candidates.items():
    selected['pipeline'].fit(train_validation[selected['features']], train_validation[target])
    test[model_name] = selected['pipeline'].predict(test[selected['features']])

validation_frames = [
    validation[['time', 'direction', target] + list(selected_candidates)].copy(),
    sarimax_validation[['time', 'direction', 'SARIMAX_no_weather']].copy()
]
test_frames = [
    test[['time', 'direction', target] + list(selected_candidates)].copy(),
    sarimax_test[['time', 'direction', 'SARIMAX_no_weather']].copy()
]
for split_name, target_frame, chunks in [('validation', validation, validation_frames), ('test', test, test_frames)]:
    merged = chunks[0].merge(chunks[1], on=['time', 'direction'], how='inner', validate='one_to_one')
    base_part = baseline_predictions.loc[baseline_predictions.split.eq(split_name)].pivot(index=['time', 'direction'], columns='model', values=['actual', 'prediction'])
    base_part.columns = [f'{actual_or_prediction}__{model_name}' for actual_or_prediction, model_name in base_part.columns]
    merged = merged.merge(base_part.reset_index(), on=['time', 'direction'], how='inner', validate='one_to_one')
    assert len(merged) == len(target_frame)
    for baseline_name in ['SeasonalNaive_24h', 'LinearRegression_A_temporal_history', 'LinearRegression_B_temporal_weather']:
        assert merged[f'actual__{baseline_name}'].notna().all()
        assert np.array_equal(merged[target].to_numpy(), merged[f'actual__{baseline_name}'].to_numpy())
    model_predictions = {
        'SeasonalNaive_24h': merged['prediction__SeasonalNaive_24h'],
        'LinearRegression_A_temporal_history': merged['prediction__LinearRegression_A_temporal_history'],
        'LinearRegression_B_temporal_weather': merged['prediction__LinearRegression_B_temporal_weather'],
        **{name: merged[name] for name in selected_candidates},
        'SARIMAX_no_weather': merged['SARIMAX_no_weather']
    }
    for prediction in model_predictions.values():
        assert prediction.notna().all() and len(prediction) == len(target_frame)
    output = merged[['time', 'direction', target]].copy()
    for name, prediction in model_predictions.items():
        output[name] = prediction.to_numpy()
    if split_name == 'validation':
        validation_output = output
    else:
        test_output = output

validation_models = [column for column in validation_output.columns if column not in {'time', 'direction', target}]
test_models = [column for column in test_output.columns if column not in {'time', 'direction', target}]
assert validation_models == test_models
validation_metrics = pd.DataFrame([{'model': name, **metric_values(validation_output[target], validation_output[name])} for name in validation_models])
selected_model = validation_metrics.sort_values(['MAE', 'RMSE']).iloc[0]['model']
test_metrics = pd.DataFrame([{'model': name, **metric_values(test_output[target], test_output[name])} for name in test_models])
assert validation_metrics.n.nunique() == 1 and test_metrics.n.nunique() == 1

# Independently recompute metrics and assert exact common timestamp-direction coverage.
for frame, table in [(validation_output, validation_metrics), (test_output, test_metrics)]:
    assert not frame.duplicated(['time', 'direction']).any()
    for name in validation_models:
        actual_values = frame[target].to_numpy()
        prediction_values = frame[name].to_numpy()
        assert np.isfinite(prediction_values).all()
        errors = prediction_values - actual_values
        nonzero = actual_values != 0
        saved = table.loc[table.model.eq(name)].iloc[0]
        assert np.isclose(np.mean(np.abs(errors)), saved.MAE)
        assert np.isclose(np.sqrt(np.mean(errors ** 2)), saved.RMSE)
        assert np.isclose(np.mean(np.abs(errors[nonzero] / actual_values[nonzero])) * 100, saved.MAPE_pct)

metrics = pd.concat([validation_metrics.assign(split='validation'), test_metrics.assign(split='test')], ignore_index=True)
weather_pairs = [
    ('LinearRegression_A_temporal_history', 'LinearRegression_B_temporal_weather'),
    ('RandomForest_A_temporal_history', 'RandomForest_B_plus_weather'),
    ('HistGradientBoosting_A_temporal_history', 'HistGradientBoosting_B_plus_weather')
]
weather_comparison = []
for split_name, table in [('validation', validation_metrics), ('test', test_metrics)]:
    for model_a_name, model_b_name in weather_pairs:
        model_a = table.loc[table.model.eq(model_a_name)].iloc[0]
        model_b = table.loc[table.model.eq(model_b_name)].iloc[0]
        weather_comparison.append({
            'split': split_name, 'family': model_a_name.removesuffix('_A_temporal_history'),
            'MAE_A': model_a.MAE, 'MAE_B': model_b.MAE, 'MAE_change_B_minus_A': model_b.MAE-model_a.MAE,
            'RMSE_A': model_a.RMSE, 'RMSE_B': model_b.RMSE, 'RMSE_change_B_minus_A': model_b.RMSE-model_a.RMSE
        })
weather_comparison = pd.DataFrame(weather_comparison)
predictions = pd.concat([
    validation_output.assign(split='validation'), test_output.assign(split='test')
], ignore_index=True).melt(
    id_vars=['split', 'time', 'direction', target], var_name='model', value_name='prediction'
).rename(columns={target: 'actual'})
assert predictions.prediction.notna().all()
assert not predictions.duplicated(['split', 'model', 'time', 'direction']).any()
assert predictions.groupby(['split', 'model']).size().eq(len(validation)).all()

metrics_path = data_dir / 'advanced_forecasting_metrics.csv'
predictions_path = data_dir / 'advanced_forecasting_predictions.csv'
weather_path = data_dir / 'advanced_forecasting_weather_increment.csv'
metrics.to_csv(metrics_path, index=False)
predictions.to_csv(predictions_path, index=False)
weather_comparison.to_csv(weather_path, index=False)
print('Validation ranking (selection only):')
display(validation_metrics.sort_values(['MAE', 'RMSE']))
print('Test evaluation:')
display(test_metrics.sort_values(['MAE', 'RMSE']))
print('Shared keys per model:', len(validation_output), 'validation and', len(test_output), 'test')
print('Weather increments B minus A (negative is improvement):')
display(weather_comparison)
print('Saved:', metrics_path, predictions_path, weather_path)

Validation ranking (selection only):


,model,n,n_nonzero_mape,MAE,RMSE,MAPE_pct
6,RandomForest_B_plus_weather,810,810,5778.637351,10106.005026,4.735941
5,RandomForest_A_temporal_history,810,810,5849.316393,10107.701423,4.676647
3,HistGradientBoosting_A_temporal_history,810,810,6271.642787,9789.255688,77.783335
4,HistGradientBoosting_B_plus_weather,810,810,6751.119987,10088.203082,46.240097
1,LinearRegression_A_temporal_history,810,810,14930.636897,19941.441927,981.605613
2,LinearRegression_B_temporal_weather,810,810,15151.819070,19858.557215,886.176941
0,SeasonalNaive_24h,810,810,15899.324691,33482.029178,10.252179
7,SARIMAX_no_weather,810,810,30022.450848,40722.975055,2887.881079


Test evaluation:


,model,n,n_nonzero_mape,MAE,RMSE,MAPE_pct
5,RandomForest_A_temporal_history,810,810,4449.933389,7127.348612,4.902390
3,HistGradientBoosting_A_temporal_history,810,810,4702.928558,6965.695302,26.107427
6,RandomForest_B_plus_weather,810,810,4794.907553,7804.432450,4.897345
4,HistGradientBoosting_B_plus_weather,810,810,5112.383317,7569.095621,27.007038
2,LinearRegression_B_temporal_weather,810,810,15861.764066,21788.450642,879.800092
1,LinearRegression_A_temporal_history,810,810,15870.571792,21703.634552,884.926160
0,SeasonalNaive_24h,810,810,22585.714815,42670.234333,16.299590
7,SARIMAX_no_weather,810,810,33665.682983,44273.315186,2392.918335


Shared keys per model: 810 validation and 810 test
Weather increments B minus A (negative is improvement):


,split,family,MAE_A,MAE_B,MAE_change_B_minus_A,RMSE_A,RMSE_B,RMSE_change_B_minus_A
0,validation,LinearRegression,14930.636897,15151.819070,221.182173,19941.441927,19858.557215,-82.884712
1,validation,RandomForest,5849.316393,5778.637351,-70.679042,10107.701423,10106.005026,-1.696397
2,validation,HistGradientBoosting,6271.642787,6751.119987,479.477200,9789.255688,10088.203082,298.947394
3,test,LinearRegression,15870.571792,15861.764066,-8.807726,21703.634552,21788.450642,84.816091
4,test,RandomForest,4449.933389,4794.907553,344.974165,7127.348612,7804.432450,677.083837
5,test,HistGradientBoosting,4702.928558,5112.383317,409.454758,6965.695302,7569.095621,603.400320


Saved: /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/advanced_forecasting_metrics.csv /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/advanced_forecasting_predictions.csv /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/advanced_forecasting_weather_increment.csv


In [11]:
# Persist the already-fitted Stage 7 Random Forest pipelines without refitting.
import json
import joblib
import sklearn
import sys

models_dir = project_root / 'Models'
models_dir.mkdir(parents=True, exist_ok=True)
rf_artifacts = {
    'RandomForest_A_temporal_history': 'random_forest_temporal_history.joblib',
    'RandomForest_B_plus_weather': 'random_forest_temporal_weather.joblib'
}
train_validation = pd.concat([train, validation], ignore_index=True).sort_values(['time', 'direction']).reset_index(drop=True)
stage7_saved_metrics = pd.read_csv(data_dir / 'advanced_forecasting_metrics.csv')
stage7_saved_predictions = pd.read_csv(data_dir / 'advanced_forecasting_predictions.csv', parse_dates=['time'])
validation_metrics_saved = stage7_saved_metrics.loc[stage7_saved_metrics.split.eq('validation')]
overall_validation_winner = validation_metrics_saved.sort_values(['MAE', 'RMSE']).iloc[0]['model']
manifest = {
    'stage': 7,
    'target': target,
    'fit_data': ['metro_weather_train_prepared.csv', 'metro_weather_validation_prepared.csv'],
    'fit_rows': int(len(train_validation)),
    'fit_split': 'train+validation after validation-only hyperparameter selection; test excluded',
    'validation_model_selection': {
        'criterion': 'minimum validation MAE; RMSE tie-break',
        'overall_selected_model': overall_validation_winner,
        'models': {}
    },
    'models': {},
    'versions': {
        'python': sys.version.split()[0],
        'scikit_learn': sklearn.__version__,
        'joblib': joblib.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__
    }
}
for model_name, filename in rf_artifacts.items():
    if model_name not in selected_candidates:
        raise RuntimeError(f'Already-fitted Stage 7 model is unavailable: {model_name}; refusing to refit implicitly.')
    selected = selected_candidates[model_name]
    feature_list = list(selected['features'])
    fitted_pipeline = selected['pipeline']
    if list(fitted_pipeline.feature_names_in_) != feature_list or not hasattr(fitted_pipeline.named_steps['model'], 'estimators_'):
        raise RuntimeError(f'Fitted pipeline or feature schema is unavailable/invalid for {model_name}.')
    saved_rows = stage7_saved_predictions.loc[
        stage7_saved_predictions.split.eq('test') & stage7_saved_predictions.model.eq(model_name)
    ].sort_values(['time', 'direction']).reset_index(drop=True)
    test_rows = test.sort_values(['time', 'direction']).reset_index(drop=True)
    if not saved_rows[['time', 'direction']].equals(test_rows[['time', 'direction']]):
        raise RuntimeError(f'Saved test keys do not align for {model_name}.')
    original_predictions = fitted_pipeline.predict(test_rows[feature_list])
    if not np.allclose(original_predictions, saved_rows['prediction'].to_numpy(), rtol=1e-12, atol=1e-8):
        raise RuntimeError(f'In-memory fitted object does not reproduce saved Stage 7 predictions: {model_name}.')
    model_path = models_dir / filename
    joblib.dump(fitted_pipeline, model_path)
    if not model_path.is_file() or model_path.stat().st_size == 0:
        raise RuntimeError(f'Model persistence failed: {model_path}')
    validation_row = tree_validation_metrics.loc[tree_validation_metrics.model.eq(model_name)].iloc[0]
    manifest['validation_model_selection']['models'][model_name] = {
        'selected_option': int(selected['option']),
        'validation_n': int(validation_row['n']),
        'validation_MAE': float(validation_row['MAE']),
        'validation_RMSE': float(validation_row['RMSE']),
        'validation_MAPE_pct': float(validation_row['MAPE_pct'])
    }
    manifest['models'][model_name] = {
        'filename': filename,
        'features': feature_list,
        'feature_order': feature_list,
        'estimator_parameters': fitted_pipeline.named_steps['model'].get_params(deep=False)
    }
manifest_path = models_dir / 'stage7_random_forest_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, default=lambda value: value.item() if hasattr(value, 'item') else str(value)) + '\n')
if not manifest_path.is_file() or manifest_path.stat().st_size == 0:
    raise RuntimeError(f'Manifest persistence failed: {manifest_path}')
print('Persisted existing fitted pipelines; saved Stage 7 test predictions were reproduced before saving:')
for model_name, filename in rf_artifacts.items():
    print(models_dir / filename)
print(manifest_path)
print('Validation-selected overall Stage 7 model:', overall_validation_winner)

Persisted existing fitted pipelines; saved Stage 7 test predictions were reproduced before saving:
/Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/Models/random_forest_temporal_history.joblib
/Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/Models/random_forest_temporal_weather.joblib
/Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/Models/stage7_random_forest_manifest.json
Validation-selected overall Stage 7 model: RandomForest_B_plus_weather


In [12]:
# Reload both artifacts in a fresh Python process and compare on identical held-out inputs.
import subprocess

probe = test.sort_values(['time', 'direction']).reset_index(drop=True).head(32).copy()
validation_script = r'''
import json, sys
import joblib
import numpy as np
import pandas as pd
models_dir, test_csv, count = sys.argv[1], sys.argv[2], int(sys.argv[3])
with open(f'{models_dir}/stage7_random_forest_manifest.json') as stream:
    manifest = json.load(stream)
rows = pd.read_csv(test_csv, parse_dates=['time']).sort_values(['time', 'direction']).reset_index(drop=True).head(count)
result = {}
for name, metadata in manifest['models'].items():
    model = joblib.load(f"{models_dir}/{metadata['filename']}")
    features = metadata['features']
    if list(model.feature_names_in_) != features or metadata['feature_order'] != features:
        raise RuntimeError(f'Feature schema mismatch: {name}')
    if not set(features).issubset(rows.columns):
        raise RuntimeError(f'Missing input features: {name}')
    values = np.asarray(model.predict(rows[features]), dtype=float)
    if not np.isfinite(values).all():
        raise RuntimeError(f'Nonfinite reloaded predictions: {name}')
    result[name] = values.tolist()
print(json.dumps(result))
'''
process = subprocess.run(
    [sys.executable, '-c', validation_script, str(models_dir), str(data_dir / 'metro_weather_test_prepared.csv'), str(len(probe))],
    check=True, capture_output=True, text=True
)
reloaded_predictions = json.loads(process.stdout.strip().splitlines()[-1])
reload_checks = []
for model_name in rf_artifacts:
    original = np.asarray(selected_candidates[model_name]['pipeline'].predict(probe[selected_candidates[model_name]['features']]), dtype=float)
    reloaded = np.asarray(reloaded_predictions[model_name], dtype=float)
    if not np.allclose(original, reloaded, rtol=1e-12, atol=1e-8):
        raise RuntimeError(f'Fresh-process prediction mismatch: {model_name}')
    reload_checks.append({'model': model_name, 'probe_rows': len(probe), 'max_abs_difference': float(np.max(np.abs(original-reloaded))), 'feature_order_match': True, 'status': 'PASS'})
reload_validation = pd.DataFrame(reload_checks)
print('Fresh-process reload validation:')
display(reload_validation)
print('PASS: persistence, reload, schema compatibility, and prediction equivalence.')

Fresh-process reload validation:


,model,probe_rows,max_abs_difference,feature_order_match,status
0,RandomForest_A_temporal_history,32,5.820766e-11,True,PASS
1,RandomForest_B_plus_weather,32,5.820766e-11,True,PASS


PASS: persistence, reload, schema compatibility, and prediction equivalence.


## Key Observations & Conclusions

Summarize the validation-selected model, final test comparison, weather contribution, separate Entry/Exit errors, limitations, and recommended next step.

In [8]:
winner = validation_metrics.sort_values(['MAE', 'RMSE']).iloc[0]
selected_test = test_metrics.loc[test_metrics.model.eq(selected_model)].iloc[0]
test_ranked = test_metrics.sort_values(['MAE', 'RMSE'])
test_winner = test_ranked.iloc[0]
seasonal_test = test_metrics.loc[test_metrics.model.eq('SeasonalNaive_24h')].iloc[0]
linear_test = test_metrics.loc[test_metrics.model.eq('LinearRegression_B_temporal_weather')].iloc[0]
print(f"Validation winner: {winner.model} (MAE {winner.MAE:.1f}, RMSE {winner.RMSE:.1f}, MAPE {winner.MAPE_pct:.1f}%).")
print(f"Selected model test: MAE {selected_test.MAE:.1f}, RMSE {selected_test.RMSE:.1f}, MAPE {selected_test.MAPE_pct:.1f}%; test leader {test_winner.model} (MAE {test_winner.MAE:.1f}, RMSE {test_winner.RMSE:.1f}).")
print(f"Baselines on test: seasonal naive MAE/RMSE {seasonal_test.MAE:.1f}/{seasonal_test.RMSE:.1f}; linear regression + weather {linear_test.MAE:.1f}/{linear_test.RMSE:.1f}.")
print('Weather test increments (B-A):', weather_comparison.loc[weather_comparison.split.eq('test'), ['family', 'MAE_change_B_minus_A', 'RMSE_change_B_minus_A']].round(1).to_dict('records'))
for direction in ['Entry', 'Exit']:
    part = test_output.loc[test_output.direction.eq(direction)]
    actual = part[target].to_numpy()
    forecast = part[selected_model].to_numpy()
    print(direction, 'selected-model test MAE/RMSE:', round(np.mean(np.abs(forecast-actual)), 1), round(np.sqrt(np.mean((forecast-actual)**2)), 1))
print('Weather variants use observed prediction-hour weather, so those comparisons are retrospective absent archived forecasts. SARIMAX uses missing off-hour observations, not zeroes.')
print('Next: evaluate with archived weather forecasts in rolling-origin tests and assess direction-specific error stability.')

Validation winner: RandomForest_B_plus_weather (MAE 5778.6, RMSE 10106.0, MAPE 4.7%).
Selected model test: MAE 4794.9, RMSE 7804.4, MAPE 4.9%; test leader RandomForest_A_temporal_history (MAE 4449.9, RMSE 7127.3).
Baselines on test: seasonal naive MAE/RMSE 22585.7/42670.2; linear regression + weather 15861.8/21788.5.
Weather test increments (B-A): [{'family': 'LinearRegression', 'MAE_change_B_minus_A': -8.8, 'RMSE_change_B_minus_A': 84.8}, {'family': 'RandomForest', 'MAE_change_B_minus_A': 345.0, 'RMSE_change_B_minus_A': 677.1}, {'family': 'HistGradientBoosting', 'MAE_change_B_minus_A': 409.5, 'RMSE_change_B_minus_A': 603.4}]
Entry selected-model test MAE/RMSE: 4483.1 7065.4
Exit selected-model test MAE/RMSE: 5106.7 8479.3
Weather variants use observed prediction-hour weather, so those comparisons are retrospective absent archived forecasts. SARIMAX uses missing off-hour observations, not zeroes.
Next: evaluate with archived weather forecasts in rolling-origin tests and assess directio

## Stage 5 vs Stage 7 Comparability Audit

This audit compares saved outputs against the prepared split files, verifies target/key/boundary identity, and independently reproduces MAE/RMSE. It does not refit models or modify forecast datasets.

In [9]:
from pathlib import Path
import json

stage5_predictions = pd.read_csv(data_dir / 'baseline_forecasting_predictions.csv', parse_dates=['time'])
stage5_metrics = pd.read_csv(data_dir / 'baseline_forecasting_metrics.csv')
stage7_predictions = pd.read_csv(data_dir / 'advanced_forecasting_predictions.csv', parse_dates=['time'])
stage7_metrics = pd.read_csv(data_dir / 'advanced_forecasting_metrics.csv')
prepared_splits = {
    split_name: pd.read_csv(data_dir / f'metro_weather_{split_name}_prepared.csv', parse_dates=['time'])
    for split_name in ['train', 'validation', 'test']
}

# Check split boundaries, target identity/units, both directions, and unique keys.
assert prepared_splits['train'].time.max() < prepared_splits['validation'].time.min()
assert prepared_splits['validation'].time.max() < prepared_splits['test'].time.min()
for split_name, prepared in prepared_splits.items():
    assert prepared.ridership_count.notna().all()
    assert prepared.direction.value_counts().to_dict() == {'Entry': len(prepared) // 2, 'Exit': len(prepared) // 2}
    assert not prepared.duplicated(['time', 'direction']).any()
    assert prepared.groupby('time').direction.nunique().eq(2).all()

# Verify every saved prediction uses the exact prepared target at its timestamp/direction.
for predictions_frame, metrics_frame, stage_name in [
    (stage5_predictions, stage5_metrics, 'Stage 5'),
    (stage7_predictions, stage7_metrics, 'Stage 7')
]:
    assert set(predictions_frame.split) == {'validation', 'test'}
    assert predictions_frame.actual.notna().all() and predictions_frame.prediction.notna().all()
    assert not predictions_frame.duplicated(['split', 'model', 'time', 'direction']).any()
    assert set(metrics_frame.split) == {'validation', 'test'}
    for split_name in ['validation', 'test']:
        prepared = prepared_splits[split_name]
        expected = prepared[['time', 'direction', 'ridership_count']]
        part = predictions_frame[predictions_frame.split.eq(split_name)]
        expected_keys = set(map(tuple, expected[['time', 'direction']].to_numpy()))
        assert part.groupby('model').size().eq(len(expected)).all()
        for model_name, group in part.groupby('model'):
            assert set(map(tuple, group[['time', 'direction']].to_numpy())) == expected_keys
            aligned = group.merge(expected, on=['time', 'direction'], how='left', validate='one_to_one')
            assert aligned.ridership_count.notna().all()
            assert np.array_equal(aligned.actual.to_numpy(), aligned.ridership_count.to_numpy())
            error = aligned.prediction.to_numpy() - aligned.ridership_count.to_numpy()
            saved = metrics_frame.loc[metrics_frame.split.eq(split_name) & metrics_frame.model.eq(model_name)].iloc[0]
            assert np.isclose(np.mean(np.abs(error)), saved.MAE)
            assert np.isclose(np.sqrt(np.mean(error ** 2)), saved.RMSE)
    print(f'{stage_name}: target alignment and saved MAE/RMSE independently verified for all models.')

stage5_lr = stage5_predictions.query("split == 'test' and model == 'linear_regression'").sort_values(['time', 'direction']).reset_index(drop=True)
stage7_lr_a = stage7_predictions.query("split == 'test' and model == 'LinearRegression_A_temporal_history'").sort_values(['time', 'direction']).reset_index(drop=True)
stage7_lr_b = stage7_predictions.query("split == 'test' and model == 'LinearRegression_B_temporal_weather'").sort_values(['time', 'direction']).reset_index(drop=True)
stage7_rf_a = stage7_predictions.query("split == 'test' and model == 'RandomForest_A_temporal_history'").sort_values(['time', 'direction']).reset_index(drop=True)
assert stage5_lr[['time', 'direction']].equals(stage7_rf_a[['time', 'direction']])
assert stage5_lr[['time', 'direction']].equals(stage7_lr_a[['time', 'direction']])
assert stage5_lr[['time', 'direction']].equals(stage7_lr_b[['time', 'direction']])
assert np.array_equal(stage5_lr.actual, stage7_rf_a.actual)

stage5_lr_mae = float(np.mean(np.abs(stage5_lr.prediction - stage5_lr.actual)))
stage7_lr_a_mae = float(np.mean(np.abs(stage7_lr_a.prediction - stage7_lr_a.actual)))
stage7_lr_b_mae = float(np.mean(np.abs(stage7_lr_b.prediction - stage7_lr_b.actual)))
stage7_rf_a_mae = float(np.mean(np.abs(stage7_rf_a.prediction - stage7_rf_a.actual)))
root_cause_summary = pd.DataFrame([{
    'comparison': 'Stage 5 linear regression vs Stage 7 Random Forest A',
    'same_target_and_units': True,
    'same_test_keys': len(stage5_lr),
    'stage5_linear_MAE': stage5_lr_mae,
    'stage7_linear_A_MAE': stage7_lr_a_mae,
    'stage7_linear_B_MAE': stage7_lr_b_mae,
    'stage7_random_forest_A_MAE': stage7_rf_a_mae,
    'linear_vs_rf_mae_gap': stage5_lr_mae - stage7_rf_a_mae,
    'interpretation': 'Different estimator/feature specification; Stage 7 linear results remain near Stage 5, so the gap is not a target, unit, join, or sample mismatch.'
}])
root_cause_path = data_dir / 'stage5_stage7_comparability_audit.csv'
root_cause_summary.to_csv(root_cause_path, index=False)

# Static model-input checks against this notebook's executed feature definitions.
assert target not in features_A + features_B and 'time' not in features_A + features_B
assert set(features_A).issubset(features_B)
assert 'ridership_count' not in features_B
assert all(split_name in {'train', 'validation', 'test'} for split_name in prepared_splits)
print(root_cause_summary.to_string(index=False))
print('PASS: same target (ridership_count in source count units), 405 Entry + 405 Exit test cases, identical boundaries/keys, exact actual alignment, no duplicate keys, metrics reproduced, and feature sets exclude target/time.')
print('No forecast predictions or metrics required correction; wrote audit summary only:', root_cause_path)

Stage 5: target alignment and saved MAE/RMSE independently verified for all models.
Stage 7: target alignment and saved MAE/RMSE independently verified for all models.
                                          comparison  same_target_and_units  same_test_keys  stage5_linear_MAE  stage7_linear_A_MAE  stage7_linear_B_MAE  stage7_random_forest_A_MAE  linear_vs_rf_mae_gap                                                                                                                                     interpretation
Stage 5 linear regression vs Stage 7 Random Forest A                   True             810       15808.969128         15870.571792         15861.764066                 4449.933389          11359.035739 Different estimator/feature specification; Stage 7 linear results remain near Stage 5, so the gap is not a target, unit, join, or sample mismatch.
PASS: same target (ridership_count in source count units), 405 Entry + 405 Exit test cases, identical boundaries/keys, exact actual 

In [10]:
# Verify historical ridership predictors for every test row against strict prior timestamps.
history_series = {
    direction: group.set_index('time')['ridership_count'].sort_index()
    for direction, group in full_target.groupby('direction')
}
test_feature_rows = prepared_splits['test']
for row in test_feature_rows.itertuples(index=False):
    series = history_series[row.direction]
    timestamp = row.time
    for lag in [1, 2, 3, 24]:
        prior_time = timestamp - pd.Timedelta(hours=lag)
        expected = series.loc[prior_time] if prior_time in series.index else np.nan
        actual_value = getattr(row, f'ridership_lag_{lag}h')
        assert (pd.isna(expected) and pd.isna(actual_value)) or np.isclose(expected, actual_value)
    for window in [6, 24]:
        observations = series.loc[(series.index >= timestamp - pd.Timedelta(hours=window)) & (series.index < timestamp)]
        expected = observations.mean() if not observations.empty else np.nan
        actual_value = getattr(row, f'ridership_rolling_mean_{window}h')
        assert (pd.isna(expected) and pd.isna(actual_value)) or np.isclose(expected, actual_value)
assert target not in features_A + features_B and 'time' not in features_A + features_B
print(f'PASS: checked {len(test_feature_rows)} test feature rows; every target lag and rolling mean uses only same-direction observations strictly before its timestamp.')

PASS: checked 810 test feature rows; every target lag and rolling mean uses only same-direction observations strictly before its timestamp.


### Audit Conclusion: PASS

- Both stages predict the same `ridership_count` in source count units, keep Entry/Exit separate, and use identical split dates: train 2024-09-02–2025-01-05, validation 2025-01-06–2025-02-01, test 2025-02-02–2025-02-28. Each test model has the same 810 timestamp-direction keys (405 per direction); actuals align exactly.
- Saved MAE/RMSE were independently reproduced for every Stage 5 and Stage 7 model. No duplicate keys, join/target mismatch, or missing test predictions were found. Model fitting/tuning used training and validation only; test was final evaluation only.
- Test results: Stage 5 linear regression MAE/RMSE **15,808.97 / 21,902.35**; Stage 7 Random Forest A **4,449.93 / 7,127.35**. Stage 7 linear A/B are **15,870.57 / 21,703.63** and **15,861.76 / 21,788.45**.
- Root cause of the ~11,359 MAE gap: unlike models and feature specifications were being compared (Stage 5 linear regression with its broader predictor set versus Stage 7 Random Forest A with temporal/history features). Stage 7 linear scores remain close to Stage 5; the gap is a model result, not a sample, target, unit, or join error.
- Stage 7 feature checks exclude target/time, and all test ridership lags/rolling means were verified against strictly earlier same-direction observations. No leakage was found. No saved forecast or metric files required correction; only this audit and `stage5_stage7_comparability_audit.csv` were added.